# NICS Background Check Analysis: Seasonal Patterns

This notebook analyzes NICS firearm background check data to answer the question:

**Do Sept–Nov consistently show spikes compared to summer months?**

We'll examine historical data to identify seasonal patterns and determine if fall months consistently show higher activity than summer months.

## 1. Import Required Libraries

First, let's import all the necessary libraries for data analysis and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pdfplumber
import re
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("Libraries imported successfully!")

## 2. Load and Prepare Data

We'll extract monthly data from the FBI's daily data PDF to get comprehensive historical information.

In [ ]:
def extract_monthly_data_from_pdf():
    """Extract monthly totals from the FBI daily data PDF"""
    pdf_path = "pdfs/nics-checks-archive.pdf"
    monthly_data = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            if page_num == 0:  # Skip first page (cover)
                continue
                
            text = page.extract_text()
            
            # Look for year patterns
            year_match = re.search(r'Year (\d{4})', text)
            if year_match:
                year = int(year_match.group(1))
                
                # Find lines with daily data and aggregate by month
                lines = text.split('\n')
                monthly_totals = [0] * 12  # Initialize 12 months
                
                for line in lines:
                    if re.match(r'^\d{1,2}\s+[\d,\s]+$', line.strip()):
                        numbers = re.findall(r'[\d,]+', line)
                        try:
                            day_numbers = [int(num.replace(',', '')) for num in numbers[1:]]  # Skip day number
                            # Add daily values to monthly totals
                            for month_idx in range(min(12, len(day_numbers))):
                                monthly_totals[month_idx] += day_numbers[month_idx]
                        except ValueError:
                            continue
                
                # Create monthly records
                month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                              'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
                
                for month_idx, month_total in enumerate(monthly_totals):
                    if month_total > 0:  # Only include months with data
                        monthly_data.append({
                            'year': year, 
                            'month': month_idx + 1,
                            'month_name': month_names[month_idx],
                            'date': f"{year}-{month_idx+1:02d}",
                            'total': month_total
                        })
    
    return pd.DataFrame(monthly_data).sort_values(['year', 'month'])

# Load the data
print("Extracting monthly data from PDF...")
monthly_df = extract_monthly_data_from_pdf()

# Display basic info about the dataset
print(f"Data loaded: {len(monthly_df)} monthly records")
print(f"Date range: {monthly_df['year'].min()} - {monthly_df['year'].max()}")
print("\nFirst few records:")
print(monthly_df.head())

## 3. Filter Data by Seasonal Periods

Let's separate the data into fall months (Sept-Nov) and summer months (June-Aug) for comparison.

In [ ]:
# Define seasonal periods
fall_months = [9, 10, 11]  # September, October, November
summer_months = [6, 7, 8]  # June, July, August

# Filter data by seasons
fall_data = monthly_df[monthly_df['month'].isin(fall_months)].copy()
summer_data = monthly_df[monthly_df['month'].isin(summer_months)].copy()

# Add season labels
monthly_df['season'] = monthly_df['month'].apply(
    lambda x: 'Fall' if x in fall_months 
             else 'Summer' if x in summer_months 
             else 'Other'
)

print(f"Fall months data: {len(fall_data)} records")
print(f"Summer months data: {len(summer_data)} records")

# Display sample data for each season
print("\nFall months sample:")
print(fall_data.head())
print("\nSummer months sample:")
print(summer_data.head())

## 4. Calculate Monthly Statistics

Let's compute statistics for each month to identify patterns and compare seasonal averages.

In [ ]:
# Calculate statistics by month
monthly_stats = monthly_df.groupby('month')['total'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
]).round(0)

monthly_stats.index = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                      'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

print("Monthly Statistics (Background Checks):")
print(monthly_stats)

# Calculate seasonal averages
fall_avg = fall_data['total'].mean()
summer_avg = summer_data['total'].mean()

print(f"\n=== SEASONAL COMPARISON ===")
print(f"Fall (Sep-Nov) average: {fall_avg:,.0f}")
print(f"Summer (Jun-Aug) average: {summer_avg:,.0f}")
print(f"Fall vs Summer difference: {fall_avg - summer_avg:,.0f}")
print(f"Fall is {(fall_avg/summer_avg - 1)*100:.1f}% higher than Summer")

## 5. Compare Sept-Nov vs Summer Months

Let's do a year-by-year comparison to see if fall consistently outperforms summer.

In [ ]:
# Year-by-year comparison
fall_by_year = fall_data.groupby('year')['total'].sum()
summer_by_year = summer_data.groupby('year')['total'].sum()

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'year': fall_by_year.index,
    'fall_total': fall_by_year.values,
    'summer_total': summer_by_year.values
})

comparison_df['fall_higher'] = comparison_df['fall_total'] > comparison_df['summer_total']
comparison_df['difference'] = comparison_df['fall_total'] - comparison_df['summer_total']
comparison_df['pct_difference'] = (comparison_df['fall_total'] / comparison_df['summer_total'] - 1) * 100

print("Year-by-Year Comparison (Fall vs Summer totals):")
print(comparison_df)

# Summary statistics
years_fall_higher = comparison_df['fall_higher'].sum()
total_years = len(comparison_df)
consistency_pct = (years_fall_higher / total_years) * 100

print(f"\n=== CONSISTENCY ANALYSIS ===")
print(f"Years where Fall > Summer: {years_fall_higher} out of {total_years}")
print(f"Consistency rate: {consistency_pct:.1f}%")
print(f"Average difference when Fall is higher: {comparison_df[comparison_df['fall_higher']]['difference'].mean():,.0f}")
print(f"Average % increase when Fall is higher: {comparison_df[comparison_df['fall_higher']]['pct_difference'].mean():.1f}%")

## 6. Visualize Seasonal Patterns

Let's create visualizations to better understand the seasonal patterns.

In [ ]:
# Create a comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Monthly averages box plot
seasonal_data = monthly_df[monthly_df['season'].isin(['Fall', 'Summer'])]
sns.boxplot(data=seasonal_data, x='season', y='total', ax=axes[0,0])
axes[0,0].set_title('Distribution of Background Checks: Fall vs Summer')
axes[0,0].set_ylabel('Background Checks')
axes[0,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))

# 2. Monthly pattern throughout the year
monthly_avg = monthly_df.groupby('month')['total'].mean().reset_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
colors = ['lightblue' if m in [1,2,3,4,5] else 'orange' if m in summer_months else 'red' if m in fall_months else 'lightgray' 
          for m in monthly_avg['month']]

axes[0,1].bar(range(12), monthly_avg['total'], color=colors, alpha=0.7)
axes[0,1].set_title('Average Background Checks by Month')
axes[0,1].set_xticks(range(12))
axes[0,1].set_xticklabels(month_names)
axes[0,1].set_ylabel('Background Checks')
axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='orange', alpha=0.7, label='Summer (Jun-Aug)'),
                   Patch(facecolor='red', alpha=0.7, label='Fall (Sep-Nov)'),
                   Patch(facecolor='lightblue', alpha=0.7, label='Other months')]
axes[0,1].legend(handles=legend_elements, loc='upper left')

# 3. Year-by-year comparison
axes[1,0].plot(comparison_df['year'], comparison_df['fall_total'], marker='o', label='Fall (Sep-Nov)', color='red', linewidth=2)
axes[1,0].plot(comparison_df['year'], comparison_df['summer_total'], marker='s', label='Summer (Jun-Aug)', color='orange', linewidth=2)
axes[1,0].set_title('Fall vs Summer Totals by Year')
axes[1,0].set_xlabel('Year')
axes[1,0].set_ylabel('Total Background Checks')
axes[1,0].legend()
axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))

# 4. Difference over time
axes[1,1].bar(comparison_df['year'], comparison_df['difference'], 
              color=['green' if x > 0 else 'red' for x in comparison_df['difference']], alpha=0.7)
axes[1,1].set_title('Fall - Summer Difference by Year')
axes[1,1].set_xlabel('Year')
axes[1,1].set_ylabel('Difference (Fall - Summer)')
axes[1,1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

## 7. Statistical Testing for Significance

Let's perform statistical tests to determine if the differences are statistically significant.

In [ ]:
# Statistical tests
from scipy.stats import ttest_ind, mannwhitneyu

# Prepare data for statistical tests
fall_values = fall_data['total'].values
summer_values = summer_data['total'].values

# T-test (assumes normal distribution)
t_stat, t_pvalue = ttest_ind(fall_values, summer_values)

# Mann-Whitney U test (non-parametric, doesn't assume normal distribution)
u_stat, u_pvalue = mannwhitneyu(fall_values, summer_values, alternative='two-sided')

print("=== STATISTICAL TESTS ===")
print(f"\n1. Two-sample t-test:")
print(f"   t-statistic: {t_stat:.3f}")
print(f"   p-value: {t_pvalue:.6f}")
print(f"   Result: {'Significant' if t_pvalue < 0.05 else 'Not significant'} at α = 0.05")

print(f"\n2. Mann-Whitney U test (non-parametric):")
print(f"   U-statistic: {u_stat:.3f}")
print(f"   p-value: {u_pvalue:.6f}")
print(f"   Result: {'Significant' if u_pvalue < 0.05 else 'Not significant'} at α = 0.05")

# Effect size (Cohen's d)
pooled_std = np.sqrt(((len(fall_values) - 1) * np.var(fall_values, ddof=1) + 
                      (len(summer_values) - 1) * np.var(summer_values, ddof=1)) / 
                     (len(fall_values) + len(summer_values) - 2))
cohens_d = (np.mean(fall_values) - np.mean(summer_values)) / pooled_std

print(f"\n3. Effect Size (Cohen's d): {cohens_d:.3f}")
print(f"   Interpretation: {'Small' if abs(cohens_d) < 0.5 else 'Medium' if abs(cohens_d) < 0.8 else 'Large'} effect")

## 8. Summary and Conclusions

Let's summarize our findings to answer both the seasonal analysis question and Black Friday analysis.

## 9. Black Friday Analysis: Holiday vs Average Day Comparison

Now let's analyze Black Friday patterns and compare them to other major holidays and average days. Black Friday is traditionally a major shopping day that could significantly impact firearm background checks.

In [ ]:
def extract_daily_data_from_pdf():
    """Extract daily data from the FBI PDF for holiday analysis"""
    pdf_path = "pdfs/nics-checks-archive.pdf"
    daily_data = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            if page_num == 0:  # Skip first page (cover)
                continue
                
            text = page.extract_text()
            
            # Look for year patterns
            year_match = re.search(r'Year (\d{4})', text)
            if year_match:
                year = int(year_match.group(1))
                
                # Find lines with daily data
                lines = text.split('\n')
                
                for line in lines:
                    if re.match(r'^\d{1,2}\s+[\d,\s]+$', line.strip()):
                        numbers = re.findall(r'[\d,]+', line)
                        try:
                            day = int(numbers[0])
                            day_numbers = [int(num.replace(',', '')) for num in numbers[1:]]  # Skip day number
                            
                            # Create records for each month (assuming 12 columns for 12 months)
                            for month_idx, daily_total in enumerate(day_numbers[:12]):
                                if daily_total > 0:  # Only include days with data
                                    date_str = f"{year}-{month_idx+1:02d}-{day:02d}"
                                    try:
                                        # Validate the date
                                        from datetime import datetime
                                        date_obj = datetime.strptime(date_str, "%Y-%m-%d")
                                        
                                        daily_data.append({
                                            'year': year,
                                            'month': month_idx + 1,
                                            'day': day,
                                            'date': date_str,
                                            'date_obj': date_obj,
                                            'total': daily_total
                                        })
                                    except ValueError:
                                        # Skip invalid dates (like Feb 30, etc.)
                                        continue
                        except (ValueError, IndexError):
                            continue
    
    return pd.DataFrame(daily_data).sort_values(['year', 'month', 'day'])

print("Extracting daily data from PDF for holiday analysis...")
daily_df = extract_daily_data_from_pdf()

print(f"Daily data extracted: {len(daily_df)} daily records")
print(f"Date range: {daily_df['year'].min()} - {daily_df['year'].max()}")
print("\nSample daily data:")
print(daily_df.head(10))

In [ ]:
def get_black_friday_dates(start_year, end_year):
    """Calculate Black Friday dates (4th Thursday of November + 1 day)"""
    from datetime import datetime, timedelta
    import calendar
    
    black_fridays = []
    
    for year in range(start_year, end_year + 1):
        # Find the 4th Thursday of November
        # November 1st
        nov_1 = datetime(year, 11, 1)
        
        # Find first Thursday
        days_to_thursday = (3 - nov_1.weekday()) % 7
        first_thursday = nov_1 + timedelta(days=days_to_thursday)
        
        # 4th Thursday is 3 weeks later
        fourth_thursday = first_thursday + timedelta(weeks=3)
        
        # Black Friday is the day after (Friday)
        black_friday = fourth_thursday + timedelta(days=1)
        
        black_fridays.append({
            'year': year,
            'date': black_friday.strftime('%Y-%m-%d'),
            'date_obj': black_friday
        })
    
    return pd.DataFrame(black_fridays)

def get_other_holidays(start_year, end_year):
    """Get other major holidays that might affect gun sales"""
    from datetime import datetime
    
    holidays = []
    
    for year in range(start_year, end_year + 1):
        # Christmas Day
        holidays.append({
            'year': year,
            'holiday': 'Christmas',
            'date': f'{year}-12-25',
            'date_obj': datetime(year, 12, 25)
        })
        
        # New Year's Day
        holidays.append({
            'year': year,
            'holiday': 'New Year',
            'date': f'{year}-01-01',
            'date_obj': datetime(year, 1, 1)
        })
        
        # Independence Day
        holidays.append({
            'year': year,
            'holiday': 'Independence Day',
            'date': f'{year}-07-04',
            'date_obj': datetime(year, 7, 4)
        })
        
        # Veterans Day
        holidays.append({
            'year': year,
            'holiday': 'Veterans Day',
            'date': f'{year}-11-11',
            'date_obj': datetime(year, 11, 11)
        })
    
    return pd.DataFrame(holidays)

# Get holiday dates
year_range = (daily_df['year'].min(), daily_df['year'].max())
black_fridays = get_black_friday_dates(year_range[0], year_range[1])
other_holidays = get_other_holidays(year_range[0], year_range[1])

print("Black Friday dates (sample):")
print(black_fridays.head())
print(f"\nTotal Black Fridays: {len(black_fridays)}")

print("\nOther holidays (sample):")
print(other_holidays.head())
print(f"\nTotal other holiday records: {len(other_holidays)}")

In [ ]:
# Match holiday dates with actual data
def get_holiday_data(daily_df, holiday_dates, holiday_name):
    """Extract background check data for specific holiday dates"""
    holiday_data = []
    
    for _, holiday in holiday_dates.iterrows():
        # Find matching daily data
        matching_data = daily_df[daily_df['date'] == holiday['date']]
        
        if not matching_data.empty:
            holiday_data.append({
                'year': holiday['year'],
                'holiday': holiday_name,
                'date': holiday['date'],
                'total': matching_data['total'].iloc[0]
            })
        else:
            # If exact date not found, it might be a weekend or data issue
            holiday_data.append({
                'year': holiday['year'],
                'holiday': holiday_name,
                'date': holiday['date'],
                'total': 0  # or np.nan
            })
    
    return pd.DataFrame(holiday_data)

# Extract Black Friday data
bf_data = get_holiday_data(daily_df, black_fridays, 'Black Friday')

# Extract other holiday data
holiday_data_list = []
for holiday_name in other_holidays['holiday'].unique():
    holiday_subset = other_holidays[other_holidays['holiday'] == holiday_name]
    holiday_data = get_holiday_data(daily_df, holiday_subset, holiday_name)
    holiday_data_list.append(holiday_data)

all_holidays_data = pd.concat([bf_data] + holiday_data_list, ignore_index=True)

# Calculate average daily background checks (excluding zeros)
daily_avg = daily_df[daily_df['total'] > 0]['total'].mean()
daily_median = daily_df[daily_df['total'] > 0]['total'].median()

print("=== HOLIDAY ANALYSIS ===")
print(f"Average daily background checks: {daily_avg:,.0f}")
print(f"Median daily background checks: {daily_median:,.0f}")

print(f"\nBlack Friday data (sample):")
print(bf_data.head())

print(f"\nHoliday statistics:")
holiday_stats = all_holidays_data.groupby('holiday')['total'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
]).round(0)
print(holiday_stats)

In [ ]:
# Compare Black Friday to average days and other holidays
print("=== BLACK FRIDAY vs AVERAGE DAY COMPARISON ===")

# Filter out zero values for meaningful comparison
bf_valid = bf_data[bf_data['total'] > 0]
daily_valid = daily_df[daily_df['total'] > 0]

if not bf_valid.empty:
    bf_avg = bf_valid['total'].mean()
    bf_median = bf_valid['total'].median()
    
    print(f"Black Friday average: {bf_avg:,.0f}")
    print(f"Average day: {daily_avg:,.0f}")
    print(f"Black Friday vs Average: {(bf_avg/daily_avg - 1)*100:.1f}% difference")
    
    print(f"\nBlack Friday median: {bf_median:,.0f}")
    print(f"Daily median: {daily_median:,.0f}")
    print(f"Black Friday vs Median: {(bf_median/daily_median - 1)*100:.1f}% difference")
    
    # Statistical test: Black Friday vs random sample of regular days
    regular_days_sample = daily_valid.sample(min(1000, len(daily_valid)), random_state=42)['total'].values
    bf_values = bf_valid['total'].values
    
    if len(bf_values) > 1:
        from scipy.stats import ttest_ind
        t_stat, p_value = ttest_ind(bf_values, regular_days_sample)
        print(f"\nStatistical test (Black Friday vs regular days):")
        print(f"T-statistic: {t_stat:.3f}")
        print(f"P-value: {p_value:.6f}")
        print(f"Result: {'Significant' if p_value < 0.05 else 'Not significant'} difference")
else:
    print("No valid Black Friday data found")

print("\n=== HOLIDAY COMPARISON ===")
# Compare all holidays
valid_holidays = all_holidays_data[all_holidays_data['total'] > 0]

if not valid_holidays.empty:
    holiday_comparison = valid_holidays.groupby('holiday')['total'].agg([
        'mean', 'median', 'count'
    ]).round(0)
    
    holiday_comparison['vs_avg_pct'] = ((holiday_comparison['mean'] / daily_avg - 1) * 100).round(1)
    holiday_comparison['vs_median_pct'] = ((holiday_comparison['median'] / daily_median - 1) * 100).round(1)
    
    print("Holiday vs Average Day Comparison:")
    print(holiday_comparison.sort_values('vs_avg_pct', ascending=False))

In [ ]:
# Create comprehensive holiday visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Black Friday vs Average Day box plot comparison
if not bf_valid.empty:
    # Create data for comparison
    comparison_data = pd.DataFrame([
        {'Type': 'Average Day', 'Checks': val} for val in daily_valid.sample(min(500, len(daily_valid)), random_state=42)['total']
    ] + [
        {'Type': 'Black Friday', 'Checks': val} for val in bf_valid['total']
    ])
    
    sns.boxplot(data=comparison_data, x='Type', y='Checks', ax=axes[0,0])
    axes[0,0].set_title('Black Friday vs Average Day Distribution')
    axes[0,0].set_ylabel('Background Checks')
    axes[0,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}K'))

# 2. Holiday comparison bar chart
if not valid_holidays.empty:
    holiday_means = valid_holidays.groupby('holiday')['total'].mean().sort_values(ascending=True)
    colors = ['red' if holiday == 'Black Friday' else 'skyblue' for holiday in holiday_means.index]
    
    axes[0,1].barh(range(len(holiday_means)), holiday_means.values, color=colors, alpha=0.7)
    axes[0,1].set_yticks(range(len(holiday_means)))
    axes[0,1].set_yticklabels(holiday_means.index)
    axes[0,1].set_xlabel('Average Background Checks')
    axes[0,1].set_title('Holiday Comparison (Average Background Checks)')
    axes[0,1].axvline(x=daily_avg, color='orange', linestyle='--', alpha=0.7, label='Daily Average')
    axes[0,1].legend()
    axes[0,1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}K'))

# 3. Black Friday trend over years
if not bf_valid.empty:
    axes[1,0].plot(bf_valid['year'], bf_valid['total'], marker='o', linewidth=2, markersize=6, color='red', alpha=0.8)
    axes[1,0].axhline(y=daily_avg, color='orange', linestyle='--', alpha=0.7, label='Daily Average')
    axes[1,0].set_title('Black Friday Trend Over Years')
    axes[1,0].set_xlabel('Year')
    axes[1,0].set_ylabel('Background Checks')
    axes[1,0].legend()
    axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}K'))

# 4. Percentage difference from average for each holiday
if not valid_holidays.empty:
    pct_diff = valid_holidays.groupby('holiday')['total'].mean() / daily_avg * 100 - 100
    pct_diff = pct_diff.sort_values(ascending=True)
    
    colors = ['red' if holiday == 'Black Friday' else 'green' if val > 0 else 'lightcoral' 
              for holiday, val in pct_diff.items()]
    
    axes[1,1].barh(range(len(pct_diff)), pct_diff.values, color=colors, alpha=0.7)
    axes[1,1].set_yticks(range(len(pct_diff)))
    axes[1,1].set_yticklabels(pct_diff.index)
    axes[1,1].set_xlabel('% Difference from Daily Average')
    axes[1,1].set_title('Holiday Performance vs Daily Average')
    axes[1,1].axvline(x=0, color='black', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 80)
print("BLACK FRIDAY ANALYSIS SUMMARY")
print("=" * 80)

if not bf_valid.empty and not valid_holidays.empty:
    # Key findings
    bf_avg = bf_valid['total'].mean()
    bf_vs_avg_pct = (bf_avg/daily_avg - 1) * 100
    
    print(f"\n🛒 BLACK FRIDAY FINDINGS:")
    print(f"• Black Friday average: {bf_avg:,.0f} background checks")
    print(f"• Daily average: {daily_avg:,.0f} background checks") 
    print(f"• Black Friday is {bf_vs_avg_pct:+.1f}% vs average day")
    
    # Holiday ranking
    holiday_ranking = valid_holidays.groupby('holiday')['total'].mean().sort_values(ascending=False)
    bf_rank = list(holiday_ranking.index).index('Black Friday') + 1 if 'Black Friday' in holiday_ranking.index else "N/A"
    
    print(f"\n🏆 HOLIDAY RANKING (by average background checks):")
    for i, (holiday, avg_checks) in enumerate(holiday_ranking.items(), 1):
        pct_diff = (avg_checks/daily_avg - 1) * 100
        marker = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}."
        marker += " 🛍️" if holiday == "Black Friday" else ""
        print(f"{marker} {holiday}: {avg_checks:,.0f} ({pct_diff:+.1f}% vs avg)")
    
    print(f"\n📊 KEY INSIGHTS:")
    
    # Check if Black Friday is above or below average
    if bf_vs_avg_pct > 10:
        bf_conclusion = "significantly HIGHER than average - major shopping impact"
    elif bf_vs_avg_pct > 0:
        bf_conclusion = "moderately higher than average"  
    elif bf_vs_avg_pct > -10:
        bf_conclusion = "close to average"
    else:
        bf_conclusion = "LOWER than average - possibly due to store closures"
    
    print(f"• Black Friday performance: {bf_conclusion}")
    
    # Find the highest and lowest performing holidays
    best_holiday = holiday_ranking.index[0]
    worst_holiday = holiday_ranking.index[-1]
    
    print(f"• Best performing holiday: {best_holiday}")
    print(f"• Lowest performing holiday: {worst_holiday}")
    
    # Weekend effect analysis
    print(f"\n💡 INTERPRETATION:")
    print(f"• Black Friday results may be influenced by federal background check office hours")
    print(f"• Many gun stores may be closed or have limited hours on federal holidays")
    print(f"• Background check systems may have different operating schedules on holidays")
    print(f"• The shopping aspect may be offset by administrative/operational constraints")
    
else:
    print("Insufficient data for Black Friday analysis")
    
print(f"\n📅 DATA COVERAGE:")
print(f"• Analysis period: {daily_df['year'].min()}-{daily_df['year'].max()}")
print(f"• Total daily records: {len(daily_df):,}")
print(f"• Black Friday records: {len(bf_valid)}")
print(f"• Other holiday records: {len(valid_holidays[valid_holidays['holiday'] != 'Black Friday'])}")

In [ ]:
print("=" * 80)
print("FINAL ANSWER: Do Sept–Nov consistently show spikes compared to summer months?")
print("=" * 80)

print(f"\n✅ KEY FINDINGS:")
print(f"• Fall months (Sep-Nov) average: {fall_avg:,.0f} background checks/month")
print(f"• Summer months (Jun-Aug) average: {summer_avg:,.0f} background checks/month")
print(f"• Fall is {(fall_avg/summer_avg - 1)*100:.1f}% higher than summer on average")

print(f"\n✅ CONSISTENCY:")
print(f"• Fall exceeded summer in {years_fall_higher} out of {total_years} years ({consistency_pct:.1f}%)")
print(f"• This shows {'high' if consistency_pct >= 80 else 'moderate' if consistency_pct >= 60 else 'low'} consistency")

print(f"\n✅ STATISTICAL SIGNIFICANCE:")
print(f"• T-test p-value: {t_pvalue:.6f} ({'significant' if t_pvalue < 0.05 else 'not significant'})")
print(f"• Mann-Whitney U p-value: {u_pvalue:.6f} ({'significant' if u_pvalue < 0.05 else 'not significant'})")
print(f"• Effect size (Cohen's d): {cohens_d:.3f} ({'large' if abs(cohens_d) >= 0.8 else 'medium' if abs(cohens_d) >= 0.5 else 'small'} effect)")

print(f"\n🎯 CONCLUSION:")
if consistency_pct >= 75 and (t_pvalue < 0.05 or u_pvalue < 0.05):
    conclusion = "YES - Fall months consistently show significant spikes compared to summer months"
elif consistency_pct >= 60:
    conclusion = "MOSTLY - Fall months show spikes more often than not, but not completely consistent"
else:
    conclusion = "NO - Fall months do not consistently show spikes compared to summer months"

print(f"   {conclusion}")

print(f"\n📊 PRACTICAL INTERPRETATION:")
print(f"• The fall hunting season (Sep-Nov) appears to drive increased firearm background checks")
print(f"• This likely reflects seasonal patterns in firearm purchases related to hunting activities")
print(f"• The pattern shows {'strong' if consistency_pct >= 80 else 'moderate'} consistency across the available years")